# Tests: Billing Components

Render-based assertion tests for `CheckoutCard`, `TrialBanner`, and `BillingStatusCard`.

In [ ]:
from fh_matui.billing import *
from fasthtml.common import to_xml

In [ ]:
#| hide

# === CheckoutCard Tests ===

# Test 1: Form has method=POST and correct action
html = to_xml(CheckoutCard(monthly_price=9.99, yearly_price=99.99, form_action='/billing/subscribe'))
assert 'method="POST"' in html or "method='POST'" in html or 'method="post"' in html, f"Form must use POST method. Got: {html[:200]}"
assert '/billing/subscribe' in html, "Form action must match form_action param"
print("✓ CheckoutCard: form POST + action")

In [ ]:
#| hide

# Test 2: Hidden fields present (plan_id, billing_period)
html = to_xml(CheckoutCard(monthly_price=9.99, yearly_price=99.99, plan_id='pro'))
assert 'name="plan_id"' in html, "Hidden field plan_id must be present"
assert 'value="pro"' in html, "plan_id value must match"
assert 'name="billing_period"' in html, "Hidden field billing_period must be present"
print("✓ CheckoutCard: hidden fields (plan_id, billing_period)")

In [ ]:
#| hide

# Test 3: Custom hidden_fields are rendered
html = to_xml(CheckoutCard(monthly_price=9.99, yearly_price=99.99, hidden_fields={'coupon': 'ABC', 'ref': '123'}))
assert 'name="coupon"' in html, "Custom hidden field 'coupon' must be rendered"
assert 'value="ABC"' in html, "Custom hidden field value 'ABC' must match"
assert 'name="ref"' in html, "Custom hidden field 'ref' must be rendered"
assert 'value="123"' in html, "Custom hidden field value '123' must match"
print("✓ CheckoutCard: custom hidden_fields")

In [ ]:
#| hide

# Test 4: default_period='yearly' sets correct active classes
html = to_xml(CheckoutCard(monthly_price=9.99, yearly_price=99.99, default_period='yearly'))
# The yearly div should have 'active', monthly should not
# Find the billing_period hidden field — should have value='yearly'
assert 'value="yearly"' in html, "billing_period hidden field must be 'yearly' when default_period='yearly'"
print("✓ CheckoutCard: default_period='yearly' sets correct initial state")

In [ ]:
#| hide

# Test 5: Two instances produce different uid values (no collisions)
html1 = to_xml(CheckoutCard(monthly_price=9.99, yearly_price=99.99))
html2 = to_xml(CheckoutCard(monthly_price=19.99, yearly_price=199.99))

# Extract uids from class names: checkout-XXXXXXXX-monthly
import re
uids1 = set(re.findall(r'checkout-([a-f0-9]{8})-monthly', html1))
uids2 = set(re.findall(r'checkout-([a-f0-9]{8})-monthly', html2))
assert len(uids1) == 1, f"Expected exactly 1 uid in first card, got {uids1}"
assert len(uids2) == 1, f"Expected exactly 1 uid in second card, got {uids2}"
assert uids1 != uids2, f"Two instances must have different uids: {uids1} vs {uids2}"
print("✓ CheckoutCard: unique instance IDs (no collisions)")

In [ ]:
#| hide

# Test 6: Trial text appears when trial_days > 0, absent when 0
html_trial = to_xml(CheckoutCard(monthly_price=9.99, yearly_price=99.99, trial_days=14))
html_no_trial = to_xml(CheckoutCard(monthly_price=9.99, yearly_price=99.99, trial_days=0))
assert '14-day free trial' in html_trial, "Trial callout must appear when trial_days > 0"
assert 'free trial' not in html_no_trial, "Trial callout must not appear when trial_days=0"
print("✓ CheckoutCard: trial_days conditional rendering")

In [ ]:
#| hide

# Test 7: Savings chip shows correct percentage
html = to_xml(CheckoutCard(monthly_price=10.00, yearly_price=96.00))
# 10*12 = 120, savings = 100 - (96/120)*100 = 20%
assert 'Save 20%' in html, "Savings chip must show correct percentage"

# No savings when prices are proportional
html_no_save = to_xml(CheckoutCard(monthly_price=10.00, yearly_price=120.00))
assert 'Save 0%' not in html_no_save, "No savings chip when yearly equals 12x monthly"
print("✓ CheckoutCard: savings chip calculation")

In [ ]:
#| hide

# === TrialBanner Tests ===

# Test 8: Info tone for days_remaining=14
html = to_xml(TrialBanner(days_remaining=14))
assert 'primary-container' in html, "14 days remaining should use info tone (primary-container)"
assert 'info' in html, "14 days remaining should use info icon"
assert '14 days remaining' in html, "Default message should include days count"
print("✓ TrialBanner: info tone for 14 days")

In [ ]:
#| hide

# Test 9: Warning tone for days_remaining=2
html = to_xml(TrialBanner(days_remaining=2))
assert 'error-container' in html, "2 days remaining should use warning tone (error-container)"
assert 'warning' in html, "2 days remaining should use warning icon"
print("✓ TrialBanner: warning tone for 2 days")

In [ ]:
#| hide

# Test 10: Warning tone for exactly 3 days (boundary)
html = to_xml(TrialBanner(days_remaining=3))
assert 'error-container' in html, "3 days remaining should use warning tone (<=3 threshold)"

# Test 11: Info tone for 4 days (just above boundary)
html = to_xml(TrialBanner(days_remaining=4))
assert 'primary-container' in html, "4 days remaining should use info tone"
print("✓ TrialBanner: boundary at 3 days")

In [ ]:
#| hide

# Test 12: Custom message overrides default
html = to_xml(TrialBanner(days_remaining=14, message="Almost there!"))
assert 'Almost there!' in html, "Custom message must appear"
assert '14 days remaining' not in html, "Default message must not appear when custom message provided"
print("✓ TrialBanner: custom message override")

In [ ]:
#| hide

# Test 13: CTA renders only when cta_text is non-empty
html_cta = to_xml(TrialBanner(days_remaining=5, cta_text='Upgrade', cta_href='/checkout'))
html_no_cta = to_xml(TrialBanner(days_remaining=5))
assert 'Upgrade' in html_cta, "CTA text must appear when provided"
assert '/checkout' in html_cta, "CTA href must appear when provided"
assert 'button' not in html_no_cta.lower() or 'button' in 'no-button-here', "No CTA button when cta_text is empty"
# More reliable check: no <a> with button class when no CTA
assert 'class="button' not in html_no_cta, "No button element when cta_text is empty"
print("✓ TrialBanner: CTA conditional rendering")

In [ ]:
#| hide

# Test 14: Singular 'day' when days_remaining=1
html = to_xml(TrialBanner(days_remaining=1))
assert '1 day remaining' in html, "Should use singular 'day' for 1"
assert '1 days' not in html, "Should not use plural 'days' for 1"
print("✓ TrialBanner: singular day for 1")

In [ ]:
#| hide

# === BillingStatusCard Tests ===

# Test 15: Each status renders correct default message
for status, expected_msg in [
    ('active', 'Your subscription is active.'),
    ('trialing', "You're currently on a free trial."),
    ('past_due', 'Payment failed. Please update your payment method.'),
    ('canceled', 'Your subscription has been canceled.'),
    ('checkout_pending', 'Complete checkout to activate your plan.'),
]:
    html = to_xml(BillingStatusCard(status=status))
    # Handle HTML entity encoding for apostrophes
    expected_check = expected_msg.replace("'", "&#x27;") if "'" in expected_msg else expected_msg
    assert expected_msg in html or expected_check in html, f"Status '{status}' must show: {expected_msg}. Got: {html[:300]}"
print("✓ BillingStatusCard: default messages for all statuses")

In [ ]:
#| hide

# Test 16: Custom messages dict overrides defaults
html = to_xml(BillingStatusCard(
    status='active',
    messages={'active': 'All good, captain!'}
))
assert 'All good, captain!' in html, "Custom message must override default"
assert 'Your subscription is active' not in html, "Default message must not appear when overridden"
print("✓ BillingStatusCard: custom messages override")

In [ ]:
#| hide

# Test 17: manage_url empty → no manage button
html = to_xml(BillingStatusCard(status='active', manage_url=''))
assert 'Manage Billing' not in html, "No manage button when manage_url is empty"

# Test 18: manage_url present → button rendered
html = to_xml(BillingStatusCard(status='active', manage_url='/billing/portal'))
assert 'Manage Billing' in html, "Manage button must appear when manage_url is provided"
assert '/billing/portal' in html, "Manage URL must be in href"
print("✓ BillingStatusCard: manage CTA conditional rendering")

In [ ]:
#| hide

# Test 19: plan_label and current_period_end render when provided
html = to_xml(BillingStatusCard(
    status='active',
    plan_label='Enterprise',
    current_period_end='December 31, 2026',
))
assert 'Enterprise' in html, "plan_label must be rendered"
assert 'December 31, 2026' in html, "current_period_end must be rendered"
print("✓ BillingStatusCard: plan_label and current_period_end")

In [ ]:
#| hide

# Test 20: Status chip colors
for status, expected_color in [
    ('active', 'green'),
    ('trialing', 'primary'),
    ('past_due', 'error'),
    ('canceled', 'grey'),
    ('checkout_pending', 'amber'),
]:
    html = to_xml(BillingStatusCard(status=status))
    assert expected_color in html, f"Status '{status}' chip must include color class '{expected_color}'"
print("✓ BillingStatusCard: status chip colors")

In [ ]:
#| hide

# Test 21: Unknown status renders gracefully
html = to_xml(BillingStatusCard(status='suspended'))
assert 'Suspended' in html, "Unknown status must render as title-cased chip label"
print("✓ BillingStatusCard: unknown status graceful fallback")

In [ ]:
#| hide

print("")
print("✅ All billing component tests passed!")